# 00. Dataset Metadata — GSE69914

This notebook extracts and structures the phenotypic metadata associated with the methylation dataset.
It parses the GEO Series Matrix file, standardizes and cleans sample annotations, and exports a unified phenotype table (pheno) to ensure consistent labeling, reproducibility, and downstream compatibility with the methylation matrix.

**Source: GEO accession GSE69914, platform Illumina HumanMethylation450 BeadChip (GPL13534)**


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     00-dataset-metadata-GSE69914                       ║
# ║ Description: : Extracts and standardizes phenotypic metadata,    ║
# ║                ensuring consistent sample labeling and           ║
# ║                reproducible downstream integration.              ║
# ║ Dataset(s):   GSE69914                                           ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 10-Nov-2025 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


### Libraries

In [ ]:
!pip install -q GEOparse polars pyarrow lz4

In [ ]:
import GEOparse
import polars as pl
import re
import os


## 1. Builf pheno_GSE69914


In [ ]:
# BUILD PHENO_GSE69914
# - Use GEOparse to read official GEO metadata
# - Build a polars DataFrame with:
#     sample_id, label, group, sentrix_id, slide_id, batch,
#     er, pr, her2, ki67
# - Save only lz4 compressed parquet to /kaggle/working

# 1) Download / load GSE69914 with GEOparse
gse_id = "GSE69914"
destdir = "/kaggle/working"

print(f"Download/parse {gse_id} with GEOparse...")
gse = GEOparse.get_GEO(geo=gse_id, destdir=destdir, annotate_gpl=False)
print("GSE69914 loaded.")

# 2) Extract metadata for each GSM (sample)
rows = []

for gsm_name, gsm in gse.gsms.items():
    # characteristics_ch1 is a list of strings like:
    # "status(0=normal 1=normal-adjacent 2=breast cancer 3=normal-brca1 4=cancer-brca1): 2"
    # "ChIP: 9257626003"
    # "sentrix id: 9257626003_R01C01"
    # "er: 1", "pr: 1", "her2: 0", "ki67: 1"
    chars = [c.lower() for c in gsm.metadata.get("characteristics_ch1", [])]

    status_code = None
    sentrix_id = None
    chip_id = None
    er = pr = her2 = ki67 = None

    for ch in chars:
        # status(0=normal 1=normal-adjacent 2=breast cancer 3=normal-brca1 4=cancer-brca1): X
        m_status = re.search(r"status\(0=normal[^)]*\):\s*([0-4])", ch)
        if m_status:
            status_code = int(m_status.group(1))

        # sentrix id: 9257626003_r01c01
        m_sentrix = re.search(r"sentrix id:\s*([\w\-]+)", ch)
        if m_sentrix:
            sentrix_id = m_sentrix.group(1)

        # ChIP: 9257626003  → 10-digit technical chip
        m_chip = re.search(r"chip:\s*(\d{10})", ch)
        if m_chip:
            chip_id = m_chip.group(1)

        # er/pr/her2/ki67 as 0/1
        m_er = re.search(r"\ber:\s*([01])", ch)
        if m_er:
            er = int(m_er.group(1))
        m_pr = re.search(r"\bpr:\s*([01])", ch)
        if m_pr:
            pr = int(m_pr.group(1))
        m_her2 = re.search(r"her2:\s*([01])", ch)
        if m_her2:
            her2 = int(m_her2.group(1))
        m_ki67 = re.search(r"ki67:\s*([01])", ch)
        if m_ki67:
            ki67 = int(m_ki67.group(1))

    # slide_id: prefer chip_id; if missing, extract from sentrix_id
    slide_id = chip_id
    if slide_id is None and sentrix_id is not None:
        m_slide = re.search(r"(\d{10})", sentrix_id)
        if m_slide:
            slide_id = m_slide.group(1)

    rows.append({
        "sample_id": gsm_name,
        "label": status_code,
        "sentrix_id": sentrix_id,
        "slide_id": slide_id,
        "er": er,
        "pr": pr,
        "her2": her2,
        "ki67": ki67,
    })

# 3) Build polars DataFrame and add group + batch
pheno = pl.DataFrame(rows)

# Explicit cast of label to integer
pheno = pheno.with_columns(
    pl.col("label").cast(pl.Int64)
)

# Use pl.lit(...) to avoid ambiguity between string and column
group_expr = (
    pl.when(pl.col("label") == 0).then(pl.lit("Normal"))
    .when(pl.col("label") == 1).then(pl.lit("Adjacent"))
    .when(pl.col("label") == 2).then(pl.lit("Tumor"))
    .when(pl.col("label") == 3).then(pl.lit("Normal_BRCA1"))
    .when(pl.col("label") == 4).then(pl.lit("Tumor_BRCA1"))
    .otherwise(pl.lit(None))
)

pheno = pheno.with_columns([
    group_expr.alias("group"),
    pl.col("slide_id").alias("batch")
])

# 4) Quick summary
print("Label distribution:")
print(
    pheno.select(pl.col("label").value_counts())
         .sort(by="label")
)

print("Group distribution:")
print(
    pheno.select(pl.col("group").value_counts())
         .sort(by="group")
)

print("Number of unique batches:",
      pheno.select(pl.col("batch").n_unique()).item())

print("Preview pheno:")
print(pheno.head())

# 5) Save lz4 parquet to /kaggle/working
out_path = "/kaggle/working/pheno_GSE69914_lz4.parquet"
pheno.write_parquet(out_path, compression="lz4")

print("Saved pheno parquet (lz4) to:")
print("  -", out_path)
